# 01 — Ingestion audit (Phase 0)

Читает **только готовые output-файлы** из `data/processed/model_inputs/` —
никакого production parsing здесь нет.

Label sources (`data/processed/label_sources/`) **намеренно не анализируются**
в этом notebook, чтобы не смешивать model inputs и label sources.


In [ ]:
import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODEL_INPUTS = REPO_ROOT / 'data' / 'processed' / 'model_inputs'
assert MODEL_INPUTS.is_dir(), (
    f'нет {MODEL_INPUTS}; сначала выполните dalel ingest'
)
print('Читаем только:', MODEL_INPUTS)

In [ ]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl(path: Path):
    if not path.is_file():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

documents = []
reports = []
for document_json in sorted(MODEL_INPUTS.glob('*/*/document.json')):
    documents.append(read_json(document_json))
    report_path = document_json.parent / 'ingestion_report.json'
    if report_path.is_file():
        reports.append(read_json(report_path))

docs = pd.DataFrame(documents)
reps = pd.DataFrame(reports)
print(f'Проектов на диске: {docs.project_id.nunique() if len(docs) else 0}')
print(f'Обработанных документов: {len(docs)}')
docs[['project_id', 'document_id', 'document_type', 'role', 'extraction_status',
      'parser_name', 'page_count', 'document_mode']] if len(docs) else docs

## Распределения: document_type, role, статусы

In [ ]:
if len(docs):
    display(docs.document_type.value_counts().rename('по document_type').to_frame())
    display(docs.role.value_counts().rename('по role').to_frame())
    display(docs.extraction_status.value_counts().rename('по статусу').to_frame())
    success_like = docs.extraction_status.isin(['success', 'partial']).sum()
    print(f'Parse success rate (success+partial): {success_like}/{len(docs)}'
          f' = {success_like / len(docs):.0%}')

## Страницы и режимы PDF (digital / scanned / mixed)

In [ ]:
if len(docs):
    display(docs.document_mode.value_counts().rename('document_mode').to_frame())
    print('Всего страниц:', int(docs.page_count.fillna(0).sum()))
    display(docs[['document_id', 'document_mode', 'page_count']]
            .sort_values('page_count', ascending=False).head(10))

## OCR usage

In [ ]:
if len(docs):
    ocr = pd.json_normalize(docs['ocr'])
    ocr.insert(0, 'document_id', docs.document_id.values)
    display(ocr[['document_id', 'mode', 'engine', 'engine_ran',
                 'ocr_page_count', 'candidate_pages', 'warnings']])
    print('Документов с реально выполненным OCR:', int(ocr.engine_ran.sum()))

## Таблицы, изображения, секции, fallback

In [ ]:
if len(reps):
    summary = reps[['document_id', 'pages_processed', 'ocr_pages', 'table_count',
                    'image_count', 'section_count', 'warning_count',
                    'fallback_used', 'elapsed_seconds', 'hash_unchanged']]
    display(summary)
    print('Всего сериализованных валидных таблиц:', int(reps.table_count.sum()))
    if 'skipped_empty_table_items' in reps:
        skipped = reps['skipped_empty_table_items'].fillna(0).astype(int)
        print('Пропущено пустых table items (не сериализованы):', int(skipped.sum()))
        nonzero = reps.loc[skipped > 0, 'document_id'].tolist()
        print('  документы с пропусками:', nonzero or 'нет')
    else:
        print('Пропущено пустых table items: поле отсутствует в отчётах (schema < 1.1.0)')
    print('Всего изображений:', int(reps.image_count.sum()))
    print('Fallback использован:', int(reps.fallback_used.sum()), 'раз(а)')
    assert bool(reps.hash_unchanged.all()), 'raw hash изменился — расследовать немедленно!'

## Warnings

In [ ]:
from collections import Counter
warning_counter = Counter()
for record in documents:
    for warning in record.get('warnings', []):
        warning_counter[warning[:100]] += 1
pd.DataFrame(warning_counter.most_common(20), columns=['warning', 'count'])

## Примеры текста и таблиц

In [ ]:
if len(docs):
    sample_dir = MODEL_INPUTS / docs.iloc[0].project_id / docs.iloc[0].document_id
    pages = read_jsonl(sample_dir / 'pages.jsonl')
    if pages:
        page = pages[0]
        print(f"{docs.iloc[0].document_id} — страница {page['page_number']}"
              f" ({page['char_count']} симв., ocr={page['ocr_applied']}):")
        print(page['text'][:1200])

In [ ]:
shown = False
for document_json in sorted(MODEL_INPUTS.glob('*/*/tables.jsonl')):
    tables = read_jsonl(document_json)
    if tables:
        table = max(tables, key=lambda t: t['num_rows'] * max(t['num_cols'], 1))
        print(f"{table['table_id']} — {table['num_rows']}x{table['num_cols']},"
              f" стр. {table['page_number']}")
        display(pd.DataFrame(table['cells']).head(8))
        shown = True
        break
if not shown:
    print('Таблиц пока нет')

## Документы с плохим extraction coverage

In [ ]:
rows = []
for document_json in sorted(MODEL_INPUTS.glob('*/*/document.json')):
    record = read_json(document_json)
    pages = read_jsonl(document_json.parent / 'pages.jsonl')
    if not pages:
        continue
    empty_pages = sum(1 for p in pages if p['char_count'] < 32)
    rows.append({
        'document_id': record['document_id'],
        'status': record['extraction_status'],
        'pages': len(pages),
        'empty_pages': empty_pages,
        'empty_ratio': round(empty_pages / len(pages), 2),
        'median_chars': int(pd.Series([p['char_count'] for p in pages]).median()),
    })
coverage = pd.DataFrame(rows).sort_values('empty_ratio', ascending=False) if rows else None
coverage

---
**Напоминание о leakage boundary:** этот notebook смотрит только на
`data/processed/model_inputs/`. Никогда не подключайте сюда
`label_sources/` — post-review документы не должны попадать в аналитические
признаки и датасеты моделей.
